In [0]:
%sql
SET TIME ZONE 'UTC';

CREATE TABLE IF NOT EXISTS smart_telemetry_electricity_usage_pricing.default.gold_meter_billing (
  meter_id          STRING        NOT NULL,
  reading_ts        TIMESTAMP     NOT NULL,
  bidding_zone      STRING,
  interval_start    TIMESTAMP,
  delivery_day      DATE,
  kwh_consumed      DECIMAL(18,6),
  price_eur_mwh     DECIMAL(12,4),
  price_eur_kwh     DECIMAL(16,7),
  cost_eur          DECIMAL(24,13),
  is_price_filled   BOOLEAN,
  gold_loaded_at    TIMESTAMP
);

MERGE INTO smart_telemetry_electricity_usage_pricing.default.gold_meter_billing AS t
USING (
  SELECT r.meter_id,
         r.reading_ts,
         r.tariff_grid_id                                   AS bidding_zone,
         g.interval_start,
         g.delivery_day,
         r.kwh_consumed,
         g.price                                            AS price_eur_mwh,
         CAST(CASE g.price_unit WHEN 'MWH' THEN g.price / 1000
                                WHEN 'KWH' THEN g.price END AS DECIMAL(16,7)) AS price_eur_kwh,
         CAST(r.kwh_consumed *
              CAST(CASE g.price_unit WHEN 'MWH' THEN g.price / 1000
                                     WHEN 'KWH' THEN g.price END AS DECIMAL(16,7))
              AS DECIMAL(24,13))                            AS cost_eur,
         g.is_filled                                        AS is_price_filled,
         current_timestamp()                                AS gold_loaded_at
  FROM smart_telemetry_electricity_usage_pricing.default.silver_meter_telemetry r
  JOIN smart_telemetry_electricity_usage_pricing.default.silver_price_grid g
    ON g.bidding_zone   = r.tariff_grid_id
   AND g.interval_start = timestamp_seconds(floor(unix_timestamp(r.reading_ts) / 900) * 900)
  WHERE g.price IS NOT NULL
) AS s
ON t.meter_id = s.meter_id AND t.reading_ts = s.reading_ts
WHEN MATCHED AND (t.price_eur_mwh <> s.price_eur_mwh OR t.is_price_filled <> s.is_price_filled)
  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
258000,0,0,258000


In [0]:
%sql
CREATE OR REPLACE VIEW smart_telemetry_electricity_usage_pricing.default.gold_unpriced_readings AS
SELECT r.meter_id, r.reading_ts, r.tariff_grid_id AS bidding_zone, r.kwh_consumed
FROM smart_telemetry_electricity_usage_pricing.default.silver_meter_telemetry r
LEFT ANTI JOIN smart_telemetry_electricity_usage_pricing.default.gold_meter_billing b
  ON b.meter_id = r.meter_id AND b.reading_ts = r.reading_ts;

CREATE OR REPLACE VIEW smart_telemetry_electricity_usage_pricing.default.gold_daily_meter_cost AS
SELECT meter_id, delivery_day, bidding_zone,
       SUM(kwh_consumed)                       AS total_kwh,
       ROUND(SUM(cost_eur), 2)                 AS total_cost_eur,
       COUNT(*)                                AS readings,
       SUM(CASE WHEN is_price_filled THEN 1 ELSE 0 END) AS readings_with_filled_price
FROM smart_telemetry_electricity_usage_pricing.default.gold_meter_billing
GROUP BY meter_id, delivery_day, bidding_zone;

In [0]:
%sql
SELECT
  (SELECT COUNT(*)          FROM smart_telemetry_electricity_usage_pricing.default.silver_meter_telemetry) AS silver_readings,
  (SELECT COUNT(*)          FROM smart_telemetry_electricity_usage_pricing.default.gold_meter_billing)     AS billed_readings,
  (SELECT COUNT(*)          FROM smart_telemetry_electricity_usage_pricing.default.gold_unpriced_readings) AS unpriced_readings,
  (SELECT SUM(kwh_consumed) FROM smart_telemetry_electricity_usage_pricing.default.silver_meter_telemetry) AS silver_kwh,
  (SELECT SUM(kwh_consumed) FROM smart_telemetry_electricity_usage_pricing.default.gold_meter_billing)
  + (SELECT COALESCE(SUM(kwh_consumed), 0) FROM smart_telemetry_electricity_usage_pricing.default.gold_unpriced_readings) AS billed_plus_unpriced_kwh,
  (SELECT COUNT(*) FROM smart_telemetry_electricity_usage_pricing.default.gold_meter_billing WHERE is_price_filled) AS billed_with_filled_price

silver_readings,billed_readings,unpriced_readings,silver_kwh,billed_plus_unpriced_kwh,billed_with_filled_price
450000,258000,192000,563.103197,563.103197,0
